<span style="font-size:32px;color:blue">**Preparation of Reference Annotation Models**</span>     <span style="font-size:22px; color:blue">(celltypist)</span></br>
<span style="font-size:16px;color:deepskyblue"></span></br>


<span style="font-size:32px;color:Blue">Celltypist Annotation</span>   <span style="font-size:18px; color:deepskyblue">Ejecutar en Lighting AI, PC o **maquina virtual**</span>

In [ ]:
#!pip install scanpy
# pip install celltypist

In [ ]:
# !pip install --user scikit-misc

In [17]:
import os
import gzip
import shutil
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
from scipy.io import mmread

import celltypist
from celltypist.models import Model

<span style="font-size:16px; color:salmon">*2. ORGANIZAR ARCHIVOS POR MUESTRA*</i></span>

In [30]:
# =========================================================
# 2. DIRECTORIO DE DATOS
# =========================================================

DATA_DIR = "data"

# =========================================================
# 3. DEFINIR MUESTRAS
# =========================================================

samples = {
    "GSM4675273_T59": "T59",
    "GSM4675274_T76": "T76",
    "GSM4675275_T77": "T77",
    "GSM4675276_T89": "T89",
    "GSM4675277_T90": "T90"
}

# =========================================================
# 4. CARGAR CADA MUESTRA
# =========================================================

adata_list = []

for gsm_id, sample_name in samples.items():

    print(f"\nCargando {gsm_id}")

    # -----------------------------------------------------
    # ARCHIVOS
    # -----------------------------------------------------

    matrix_file = os.path.join(
        DATA_DIR,
        f"{gsm_id}_matrix.mtx.gz"
    )

    genes_file = os.path.join(
        DATA_DIR,
        f"{gsm_id}_genes.tsv.gz"
    )

    barcodes_file = os.path.join(
        DATA_DIR,
        f"{gsm_id}_barcodes.tsv.gz"
    )

    # -----------------------------------------------------
    # LEER MATRIZ
    # -----------------------------------------------------

    X = mmread(matrix_file).T.tocsr()

    print("Matrix shape:", X.shape)

    # -----------------------------------------------------
    # LEER GENES
    # -----------------------------------------------------

    genes = pd.read_csv(
        genes_file,
        header=None,
        sep="\t"
    )

    # -----------------------------------------------------
    # LEER BARCODES
    # -----------------------------------------------------

    barcodes = pd.read_csv(
        barcodes_file,
        header=None
    )

    # -----------------------------------------------------
    # CREAR ANNDATA
    # -----------------------------------------------------

    adata = ad.AnnData(X)

    # nombres genes
    adata.var_names = genes[1].values

    # nombres células
    gsm_prefix = gsm_id.split("_")[0]

    adata.obs_names = [
        f"{gsm_prefix}@{x}"
        for x in barcodes[0].values
    ]

    adata.var_names_make_unique()

    # metadata básica
    adata.obs["Sample"] = sample_name

    print(adata)

    adata_list.append(adata)


Cargando GSM4675273_T59
Matrix shape: (16280, 33538)
AnnData object with n_obs × n_vars = 16280 × 33538
    obs: 'Sample'

Cargando GSM4675274_T76
Matrix shape: (16679, 33538)
AnnData object with n_obs × n_vars = 16679 × 33538
    obs: 'Sample'

Cargando GSM4675275_T77
Matrix shape: (8136, 33538)
AnnData object with n_obs × n_vars = 8136 × 33538
    obs: 'Sample'

Cargando GSM4675276_T89
Matrix shape: (6100, 33538)
AnnData object with n_obs × n_vars = 6100 × 33538
    obs: 'Sample'

Cargando GSM4675277_T90
Matrix shape: (4926, 33538)
AnnData object with n_obs × n_vars = 4926 × 33538
    obs: 'Sample'


<span style="font-size:16px; color:salmon">*3. CONCATENAR TODAS LAS MUESTRAS*</i></span>

In [31]:
print("\nConcatenando muestras...")

adata_ref = sc.concat(
    adata_list,
    join="outer",
    label="batch",
    keys=list(samples.values())
)

print(adata_ref)


Concatenando muestras...
AnnData object with n_obs × n_vars = 52121 × 33538
    obs: 'Sample', 'batch'


<span style="font-size:16px; color:salmon">*4. CARGAR METADATA*</i></span>

In [36]:
meta = pd.read_csv(
    "OV_GSE154600_CellMetainfo_table.tsv",
    sep="\t",
    index_col=0
)

print(meta.head())

                                  UMAP_1    UMAP_2  Cluster  \
Cell                                                          
GSM4675273@AAACCTGTCAATCACG-1  15.697671  3.701826       18   
GSM4675273@AAACGGGGTAAGTGGC-1  15.700478  3.689018       18   
GSM4675273@AAATGCCAGTCTCGGC-1  15.633898  3.690286       18   
GSM4675273@AACACGTGTGGTCCGT-1  15.833066  3.840306       18   
GSM4675273@AACTCCCTCGCATGGC-1  15.659184  3.689957       18   

                              Celltype (malignancy) Celltype (major-lineage)  \
Cell                                                                           
GSM4675273@AAACCTGTCAATCACG-1          Immune cells                        B   
GSM4675273@AAACGGGGTAAGTGGC-1          Immune cells                        B   
GSM4675273@AAATGCCAGTCTCGGC-1          Immune cells                        B   
GSM4675273@AACACGTGTGGTCCGT-1          Immune cells                        B   
GSM4675273@AACTCCCTCGCATGGC-1          Immune cells                        B  

<span style="font-size:16px; color:salmon">*5. AÑADIR METADATA A LAS CÉLULAS*</i></span>

In [38]:
common_cells = adata_ref.obs_names.intersection(meta.index)

print(len(common_cells))

42583


In [39]:
adata_ref = adata_ref[common_cells].copy()

adata_ref.obs = meta.loc[common_cells]

print(adata_ref)
print(adata_ref.obs.head())

AnnData object with n_obs × n_vars = 42583 × 33538
    obs: 'UMAP_1', 'UMAP_2', 'Cluster', 'Celltype (malignancy)', 'Celltype (major-lineage)', 'Celltype (minor-lineage)', 'Patient', 'Sample', 'Tissue'
                                 UMAP_1    UMAP_2  Cluster  \
GSM4675273@AAACCTGAGCTGCCCA-1  4.707060 -7.464179        6   
GSM4675273@AAACCTGAGGACTGGT-1 -5.581871  2.019390        7   
GSM4675273@AAACCTGAGTCATCCA-1  6.644882 -8.831549        5   
GSM4675273@AAACCTGCAAGCCCAC-1  5.007848 -7.842579        4   
GSM4675273@AAACCTGCAAGCGCTC-1 -7.341664  3.969479       11   

                              Celltype (malignancy) Celltype (major-lineage)  \
GSM4675273@AAACCTGAGCTGCCCA-1          Immune cells               Mono/Macro   
GSM4675273@AAACCTGAGGACTGGT-1         Stromal cells               Epithelial   
GSM4675273@AAACCTGAGTCATCCA-1          Immune cells               Mono/Macro   
GSM4675273@AAACCTGCAAGCCCAC-1          Immune cells               Mono/Macro   
GSM4675273@AAACCTGCAAGCGC

<span style="font-size:16px; color:salmon">*4. QC*</i></span>

In [40]:
sc.pp.filter_cells(
    adata_ref,
    min_genes=200
)

sc.pp.filter_genes(
    adata_ref,
    min_cells=3
)

<span style="font-size:16px; color:salmon">*5. Normalización*</i></span>

In [41]:
sc.pp.normalize_total(
    adata_ref,
    target_sum=1e4
)

sc.pp.log1p(adata_ref)

In [42]:
adata_ref.obs

,UMAP_1,UMAP_2,Cluster,Celltype (malignancy),Celltype (major-lineage),Celltype (minor-lineage),Patient,Sample,Tissue,n_genes
GSM4675273@AAACCTGAGCTGCCCA-1,4.707060,-7.464179,6,Immune cells,Mono/Macro,Monocyte,T59,T59,Tumor,805
GSM4675273@AAACCTGAGGACTGGT-1,-5.581871,2.019390,7,Stromal cells,Epithelial,Epithelial,T59,T59,Tumor,2561
GSM4675273@AAACCTGAGTCATCCA-1,6.644882,-8.831549,5,Immune cells,Mono/Macro,M1,T59,T59,Tumor,1031
GSM4675273@AAACCTGCAAGCCCAC-1,5.007848,-7.842579,4,Immune cells,Mono/Macro,M1,T59,T59,Tumor,1044
GSM4675273@AAACCTGCAAGCGCTC-1,-7.341664,3.969479,11,Malignant cells,Malignant,Malignant,T59,T59,Tumor,2608
...,...,...,...,...,...,...,...,...,...,...
GSM4675277@TTTGTCACATTCACTT-1,-6.717717,2.764482,2,Stromal cells,Epithelial,Epithelial,T90,T90,Tumor,2997
GSM4675277@TTTGTCACATTCTTAC-1,-5.699539,5.228629,11,Malignant cells,Malignant,Malignant,T90,T90,Tumor,3376
GSM4675277@TTTGTCAGTAGCTGCC-1,-5.836710,-8.202273,3,Stromal cells,Fibroblasts,Fibroblasts,T90,T90,Tumor,2604
GSM4675277@TTTGTCAGTTGAGGTG-1,-8.138184,-3.451313,9,Stromal cells,Fibroblasts,Fibroblasts,T90,T90,Tumor,3429


<span style="font-size:16px; color:salmon">*6. Definir labels Celltypist*</i></span>

In [43]:
adata_ref.obs["cell_type"] = (
    adata_ref.obs["Celltype (major-lineage)"]
)

<span style="font-size:16px; color:salmon">*7. Ver distribución*</i></span>

In [44]:
print(
    adata_ref.obs["cell_type"].value_counts()
)

Epithelial        8669
Mono/Macro        8333
Fibroblasts       7622
CD8T              6471
CD8Tex            4355
Malignant         2320
Plasma            1298
Endothelial       1186
Myofibroblasts    1067
B                  763
Tprolif            499
Name: cell_type, dtype: int64


In [45]:
adata_ref.write("ad_Pablo.h5ad", compression="gzip")

... storing 'Celltype (malignancy)' as categorical
... storing 'Celltype (major-lineage)' as categorical
... storing 'Celltype (minor-lineage)' as categorical
... storing 'Patient' as categorical
... storing 'Sample' as categorical
... storing 'Tissue' as categorical
... storing 'cell_type' as categorical


<span style="font-size:16px; color:salmon">*8. Entrenar modelo*</i></span>

In [47]:
model = celltypist.train(
    adata_ref,
    labels="cell_type")

🍳 Preparing data before training
🔬 Input data has 42583 cells and 24839 genes
⚖️ Scaling input data
🏋️ Training data using logistic regression
✅ Model training done!


In [48]:
model.write('C:Reference_Pablo_celltypist.pkl')